## 第 7 课：三次归约与数值稳定

题目：[Triton: Numerically Stable Fused Softmax](https://www.deep-ml.com/problems/973?from=Triton%20Essentials)（ID 973）

计算目标（逐行 softmax，数值稳定版）：

In [ ]:
y[m, n] = exp(x[m, n] - max_k x[m, k]) / sum_j exp(x[m, j] - max_k x[m, k])

整个计算必须在一个 kernel 内完成，不允许把中间结果写回全局内存再读回来。

例如（手工算一下）：x 第 0 行是 `[1, 2, 3]`，行最大值是 3：

In [ ]:
exp(1-3) + exp(2-3) + exp(3-3) = e^-2 + e^-1 + 1 ≈ 1.5032
y = [e^-2/1.5032, e^-1/1.5032, 1/1.5032]
  ≈ [0.0900, 0.2447, 0.6652]

### 1. 三次归约串起来

每行依次做三个归约，共享同一份加载数据：

In [ ]:
row = tl.load(x_ptr + pid * stride_xm + offs_n, mask=mask, other=-float('inf'))
row_max = tl.max(row, axis=0)          # 第 1 次归约：行最大值
exp_row = tl.exp(row - row_max)        # 逐元素
denom   = tl.sum(exp_row, axis=0)      # 第 2 次归约：归一化分母
y = exp_row / denom                    # 第 3 步：逐元素除法

### 2. 加载时 other 用 -inf

求最大值时，padding 的 lane 必须**永远不可能成为最大值**。如果 `other=0.0`，当整行都是负数时，0 会变成伪最大，整个 softmax 就错了：

In [ ]:
other=-float('inf')   # padding lane 的值是 -inf，永远不会被 max 选中

### 3. 求分母时不用再 mask

padding lane 的值是 `-inf`，所以：

In [ ]:
exp(-inf - row_max) = exp(-inf) = 0

它们对分母的贡献天然是 0，不需要 `tl.where(mask, ..., 0.0)` 再清一次。

### 4. 为什么减去行最大值？

`exp(x)` 在 x 很大时会溢出成 inf/NaN。减去行最大值后指数 <= 0：

In [ ]:
exp(x - max) ∈ (0, 1]

分子分母都是 (0,1] 区间的数，数学上等价、数值上稳定。

## 你的代码骨架

In [ ]:
import torch
import triton
import triton.language as tl


@triton.jit
def softmax_kernel(
    output_ptr,
    input_ptr,
    M,
    N,
    stride_xm,
    stride_ym,
    BLOCK_SIZE_N: tl.constexpr,
):
    # TODO 1：行 ID / offs_n / mask
    # 注意：加载时 other 用 -inf，而不是 0.0（为什么？见问题 1）

    # TODO 2：row_max = tl.max(row, axis=0)
    # padding lane 是 -inf，永远不会成为最大值

    # TODO 3：exp_row = tl.exp(row - row_max)
    # padding lane：exp(-inf - row_max) = 0，自然不贡献，不用重新 mask

    # TODO 4：denom = tl.sum(exp_row, axis=0)

    # TODO 5：y = exp_row / denom，带 mask 写入 output
    pass


def softmax(x: torch.Tensor) -> torch.Tensor:
    # TODO 6：取得 M、N

    # TODO 7：BLOCK_SIZE_N = triton.next_power_of_2(N)

    # TODO 8：分配 output（形状与 x 相同）

    # TODO 9：创建一维 grid（M,）

    # TODO 10：启动 kernel

    # TODO 11：返回 output
    pass

同时回答：

1. 为什么加载整行时 `other` 用 `-inf` 而不是 `0.0`？（提示：考虑整行都是负数的情况）
2. 为什么求分母时不需要再重新 mask？padding lane 的 exp 值是多少？
3. 为什么要先减去行最大值再 exp？不减去会发生什么？（提示：`exp` 的溢出）

把代码和三个答案发给我，我继续审查。